Install Dependencies

In [3]:
!pip install torch transformers scikit-learn


In [4]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [20]:
import json

# Create the intents.json content directly
intents_content = '''{
  "intents": [
    {
      "tag": "greeting",
      "patterns": [
        "Hi",
        "Hello",
        "Hey",
        "Good morning",
        "Good evening"
      ],
      "responses": [
        "Hello! How can I help you?",
        "Hi there! What can I do for you?"
      ]
    },
    {
      "tag": "admission_query",
      "patterns": [
        "How can I get admission?",
        "What is the admission process?",
        "How to apply for admission?",
        "Tell me about admission"
      ],
      "responses": [
        "You can apply online through the official website. Admission is based on merit."
      ]
    },
    {
      "tag": "fees_info",
      "patterns": [
        "What is the fee structure?",
        "How much are the fees?",
        "Tell me fees details",
        "What is the yearly fee?"
      ],
      "responses": [
        "The annual fee is ₹85,000 for B.Tech programs."
      ]
    },
    {
      "tag": "goodbye",
      "patterns": [
        "Bye",
        "Goodbye",
        "See you later",
        "Thank you, bye"
      ],
      "responses": [
        "Goodbye! Have a great day.",
        "See you soon!"
      ]
    }
  ]
}'''

# Write to file
with open("intents.json", "w") as f:
    f.write(intents_content)

# Now load it
with open("intents.json", "r") as f:
    data = json.load(f)

texts, labels = [], []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

print(f"Loaded {len(texts)} patterns with {len(set(labels))} unique labels")
print(f"\nLabels: {set(labels)}")
print(f"Sample texts: {texts[:3]}")

Loaded 17 patterns with 4 unique labels

Labels: {'greeting', 'goodbye', 'admission_query', 'fees_info'}
Sample texts: ['Hi', 'Hello', 'Hey']


In [21]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

torch.save(label_encoder.classes_, "label_classes.pt")


In [22]:
X_train, X_val, y_train, y_val = train_test_split(
    texts, encoded_labels, test_size=0.2, random_state=42
)


In [23]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [24]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [25]:
train_dataset = IntentDataset(X_train, y_train)
val_dataset = IntentDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [26]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(set(encoded_labels))
)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
model.train()
epochs = 5

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


Epoch 1/5, Loss: 2.7606
Epoch 2/5, Loss: 2.6072
Epoch 3/5, Loss: 2.2618
Epoch 4/5, Loss: 1.9593
Epoch 5/5, Loss: 1.6453


In [28]:
model.save_pretrained("intent_model")
tokenizer.save_pretrained("intent_model")


('intent_model/tokenizer_config.json',
 'intent_model/special_tokens_map.json',
 'intent_model/vocab.txt',
 'intent_model/added_tokens.json',
 'intent_model/tokenizer.json')

In [29]:
model.save_pretrained("../model/intent_model")
tokenizer.save_pretrained("../model/intent_model")
torch.save(label_encoder.classes_, "../model/label_classes.pt")
